# Side-View 7KP Body-End Pose Training — Google Colab

**Keypoints (order must match the packaged labels):**
0. `front_wheel_center`
1. `front_wheel_ground`
2. `rear_wheel_center`
3. `rear_wheel_ground`
4. `ground_ref`
5. `front_bumper`
6. `rear_bumper`

**Steps:**
1. Run §1 — check GPU
2. Run §2 — install
3. Run §3 — upload dataset zip (built by `package_7kp_colab.ps1`)
4. Run §4 — unpack
5. Run §5 — train
6. Run §6 — out-of-sample evaluation (upload holdout zip separately)
7. Run §7 — download `best.pt`

## §1 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

## §2 — Install Ultralytics

In [ ]:
%pip install -q 'ultralytics>=8.2' opencv-python-headless

## §3 — Upload Dataset Zip

Run `package_7kp_colab.ps1` locally first — it will create  
`~/Downloads/sdi_7kp_colab_dataset.zip`.  
Then upload it below.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select sdi_7kp_colab_dataset.zip
DATASET_ZIP = list(uploaded.keys())[0]
print('Uploaded:', DATASET_ZIP)

## §4 — Unpack

In [ ]:
import zipfile, pathlib, shutil, re

DATASET_DIR = pathlib.Path('/content/sdi_7kp_dataset')
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)

with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall(DATASET_DIR)

# Ensure val directory exists (Ultralytics errors if the directory is absent)
(DATASET_DIR / 'images' / 'val').mkdir(parents=True, exist_ok=True)
(DATASET_DIR / 'labels' / 'val').mkdir(parents=True, exist_ok=True)

# Rewrite YAML path to the real absolute path so Ultralytics resolves images correctly
yaml_path = DATASET_DIR / 'dataset_pose.yaml'
yaml_text = yaml_path.read_text()
yaml_text = re.sub(r'^\s*path:.*$', f'path: {DATASET_DIR}', yaml_text, flags=re.MULTILINE)
yaml_path.write_text(yaml_text)

# Verify
train_imgs = list((DATASET_DIR / 'images' / 'train').glob('*.jpg')) + \
             list((DATASET_DIR / 'images' / 'train').glob('*.png'))
val_imgs   = list((DATASET_DIR / 'images' / 'val').glob('*.jpg'))   + \
             list((DATASET_DIR / 'images' / 'val').glob('*.png'))
train_lbls = list((DATASET_DIR / 'labels' / 'train').glob('*.txt'))
val_lbls   = list((DATASET_DIR / 'labels' / 'val').glob('*.txt'))

print(f'Train images : {len(train_imgs)}')
print(f'Train labels : {len(train_lbls)}')
print(f'Val   images : {len(val_imgs)}')
print(f'Val   labels : {len(val_lbls)}')

print('\ndataset_pose.yaml (after path fix):')
print(yaml_path.read_text())

DATASET_YAML = str(yaml_path)


## §5 — Train

Key changes vs the local CPU smoke run:
- `imgsz=640` (vs 384 on CPU)
- `batch=16` (vs 2 on CPU)
- `epochs=150` — early-stopping (`patience=30`) will exit before if val stabilises
- `device=0` (T4 GPU)
- `augment=True`, mosaic+flipud enabled for better generalisation on this small dataset

In [ ]:
from ultralytics import YOLO
import pathlib

# DATASET_YAML is set at the end of section 4 (after path fix)
# If running this cell standalone, uncomment:
# DATASET_YAML = '/content/sdi_7kp_dataset/dataset_pose.yaml'
RUN_NAME      = 'side_view_7kp_colab_gpu'
PROJECT       = '/content/runs'

model = YOLO('yolov8n-pose.pt')  # nano pose — good for small dataset

results = model.train(
    data      = DATASET_YAML,
    task      = 'pose',
    epochs    = 150,
    patience  = 30,       # stop if no val improvement for 30 epochs
    imgsz     = 640,
    batch     = 16,
    device    = 0,        # T4 GPU
    optimizer = 'AdamW',
    cos_lr    = True,
    lr0       = 0.001,
    lrf       = 0.01,
    weight_decay = 0.0005,
    warmup_epochs = 5,
    # Augmentation — helps generalise on small dataset
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    degrees   = 0.0,      # no rotation — side-view must stay upright
    translate = 0.1,
    scale     = 0.5,
    flipud    = 0.0,      # no vertical flip
    fliplr    = 0.5,      # horizontal flip is valid (left/right-looking cars)
    mosaic    = 0.5,      # partial mosaic helps with small count
    close_mosaic = 10,
    # Reproducibility
    seed      = 42,
    deterministic = True,
    amp       = True,     # mixed precision — faster on T4
    project   = PROJECT,
    name      = RUN_NAME,
    exist_ok  = True,
)

BEST_PT = pathlib.Path(PROJECT) / RUN_NAME / 'weights' / 'best.pt'
print('\nbest.pt at:', BEST_PT)
print('best.pt exists:', BEST_PT.exists())

## §6 — Out-of-Sample Holdout Evaluation

Upload `sdi_7kp_holdout.zip` (33 holdout side-view images with no labels).  
Create it locally:
```powershell
Compress-Archive -Path d:\project\sdi-helper\yolo_training\runs\side_view_pose_7kp_bumper_oos_20260524\images\* `
                 -DestinationPath $env:USERPROFILE\Downloads\sdi_7kp_holdout.zip
```
The cell below runs inference and saves an annotated contact sheet.

In [ ]:
from google.colab import files as colab_files
import zipfile, pathlib, shutil

uploaded_oos = colab_files.upload()  # select sdi_7kp_holdout.zip
HOLDOUT_ZIP = list(uploaded_oos.keys())[0]

HOLDOUT_DIR = pathlib.Path('/content/holdout_images')
if HOLDOUT_DIR.exists():
    shutil.rmtree(HOLDOUT_DIR)
HOLDOUT_DIR.mkdir()

with zipfile.ZipFile(HOLDOUT_ZIP, 'r') as z:
    z.extractall(HOLDOUT_DIR)

imgs = list(HOLDOUT_DIR.rglob('*.jpg')) + list(HOLDOUT_DIR.rglob('*.png'))
print(f'Holdout images: {len(imgs)}')

In [ ]:
from ultralytics import YOLO
import pathlib, json

BEST_PT     = pathlib.Path('/content/runs/side_view_7kp_colab_gpu/weights/best.pt')
HOLDOUT_DIR = pathlib.Path('/content/holdout_images')
OOS_OUT     = pathlib.Path('/content/oos_results')

model = YOLO(str(BEST_PT))

results = model.predict(
    source  = str(HOLDOUT_DIR),
    conf    = 0.25,
    iou     = 0.7,
    device  = 0,
    save    = True,
    save_txt= True,
    project = str(OOS_OUT.parent),
    name    = OOS_OUT.name,
    exist_ok= True,
)

# ── Geometry sanity gate (mirrors the local holdout gate) ─────────────────
KP = ['front_wheel_center','front_wheel_ground','rear_wheel_center',
      'rear_wheel_ground','ground_ref','front_bumper','rear_bumper']
FWC, FWG, RWC, RWG, GND, FB, RB = 0, 1, 2, 3, 4, 5, 6

pass_count = 0
fail_reasons = {}

for r in results:
    name = pathlib.Path(r.path).stem
    if r.keypoints is None or len(r.keypoints.xy) == 0:
        fail_reasons[name] = ['no_detection']
        continue
    kps = r.keypoints.xy[0].cpu().numpy()  # shape (7, 2)
    if kps.shape[0] < 7:
        fail_reasons[name] = [f'only_{kps.shape[0]}_kps']
        continue

    fw_cx = kps[FWC][0]; rw_cx = kps[RWC][0]
    fb_x  = kps[FB][0];  rb_x  = kps[RB][0]

    # Determine facing direction from wheel positions
    facing_right = fw_cx < rw_cx  # front wheel left of rear wheel

    fails = []
    if facing_right:
        if not (fb_x < fw_cx):  fails.append('front_endpoint_inside_body')
        if not (rb_x > rw_cx):  fails.append('rear_endpoint_inside_body')
    else:
        if not (fb_x > fw_cx):  fails.append('front_endpoint_inside_body')
        if not (rb_x < rw_cx):  fails.append('rear_endpoint_inside_body')

    if fails:
        fail_reasons[name] = fails
    else:
        pass_count += 1

total = len(results)
print(f'\n=== Holdout gate: {pass_count}/{total} PASS ===')
print(f'Fail breakdown:')
from collections import Counter
flat_fails = [f for v in fail_reasons.values() for f in v]
for reason, count in Counter(flat_fails).most_common():
    print(f'  {reason}: {count}')

if pass_count >= int(total * 0.7):
    print('\n✅ PROMOTE — ≥70% holdout pass. Ready for backend integration.')
else:
    print('\n❌ REJECT — <70% holdout pass. Do not promote.')

# Save gate report
report = {'pass': pass_count, 'total': total, 'fail_reasons': fail_reasons}
with open('/content/holdout_gate_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('\nReport saved to /content/holdout_gate_report.json')

## §7 — Download best.pt and gate report

In [ ]:
from google.colab import files as colab_files
import pathlib

BEST_PT = pathlib.Path('/content/runs/side_view_7kp_colab_gpu/weights/best.pt')
REPORT  = pathlib.Path('/content/holdout_gate_report.json')

colab_files.download(str(BEST_PT))
if REPORT.exists():
    colab_files.download(str(REPORT))